In [1]:
# use the regression RF model exported in 5_1 to predict complete monthly rasters 

# 1. Load all predictor rasters for the month
# 2. Stack and create a df
# 3. Format df to fit column requirement of the model
# 4. Make predictions in the df of precipitation_sum
# 5. Make a raster based on the predicted precipitation
# 6. Saved the month's predicted precipitation raster

In [ ]:
import os
import gc
import glob
import numpy as np
import pandas as pd 
import xarray as xr 
import rioxarray as rxr 
import pyarrow as pa    
import pyarrow.parquet as pq    
from tqdm import tqdm

START_YEAR, START_MONTH = 2001, 1
END_YEAR,   END_MONTH   = 2005, 12

# Paths — monthly raster aggregation outputs
MONTHLY_RASTER_DIR = r"..\Data\Processed\RegressionModel\Monthly_Rasters"
TEMPLATE_PATH      = r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif"

In [ ]:
# Static raster paths (same as daily pipeline)
STATIC_PATHS = {
    "cluster":         r"..\Data\Processed\Spatial_clustering\Precipitation_clusters.tif",
    "elev_mean":       r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif",
    "elev_std":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif",
    "elev_max":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif",
    "elev_min":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif",
    "slope_mean":      r"..\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif",
    "aspect_mean":     r"..\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif",
    "elev_smoothed":   r"..\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif",
    "fetch_1000m_dist":r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif",
    "fetch_1000m_dir": r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif",
    "fetch_500m_dist": r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif",
    "fetch_500m_dir":  r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif",
    "local_relief":    r"..\Data\Processed\Fetch\Preprocessing\local_relief.tif",
}

# Output directories
OUT_BASE           = r"..\Data\Processed\PredictedRasters"
MONTHLY_PRED_DIR  = os.path.join(OUT_BASE, "monthly_dynamic_pred_df")
PARQUET_PRED_DIR  = os.path.join(OUT_BASE, "yearly_parquet_pred")
STATIC_DIR         = os.path.join(OUT_BASE, "static_df")
 
for d in [MONTHLY_PRED_DIR, PARQUET_PRED_DIR,
          STATIC_DIR, OUT_BASE]:
    os.makedirs(d, exist_ok=True)
 
# Template 
template = rxr.open_rasterio(TEMPLATE_PATH, masked=True)

In [3]:
# Helper - month label
def mlabel(year: int, month: int) -> str:
    return f"{year}{month:02d}"

# Helper - month cyclic
def month_cyclical(month: int):
    """Return (sin, cos) encoding for a calendar month (1–12)."""
    angle = 2 * np.pi * month / 12
    return np.sin(angle), np.cos(angle)

# build static features dataframe
static_csv = os.path.join(STATIC_DIR, 'static_df.csv')

if os.path.exists(static_csv):
    print("Loading saved static df")
    static_df = pd.read_csv(static_csv)
else:
    print("Buidling static df from rasters")
    aligned = []
    for name, path in STATIC_PATHS.items():
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        aligned.append(da)
    static_xr = xr.merge(aligned, compat = 'override')
    static_df = static_xr.to_dataframe().reset_index()
    static_df.to_csv(static_csv, index = False)
    del aligned, static_xr
    gc.collect()

print(f"static_df: {len(static_df):,} pixels")

Buidling static df from rasters
static_df: 484,619 pixels


In [4]:
#Aggregate dynamic rasters 

def build_monthly_paths(year: int, month: int) -> dict:
    """
    Return dict of {feature_name: filepath} for all monthly rasters
    for the given year-month. Missing files are noted but not fatal —
    the loader handles absent paths with NaN fill.
    """
    L = mlabel(year, month)
    R = MONTHLY_RASTER_DIR
    return {
        # Precipitation sums — all five products
        "era5l_monthly_sum":        f"{R}/era5l/era5l_monthly_sum_{L}.tif",
        "chirps_monthly_sum":       f"{R}/chirps/chirps_monthly_sum_{L}.tif",
        "har_monthly_sum":          f"{R}/har/har_monthly_sum_{L}.tif",
        "persiann_monthly_sum":     f"{R}/persiann/persiann_monthly_sum_{L}.tif",
        "mswep_monthly_sum":        f"{R}/mswep/mswep_monthly_sum_{L}.tif",
        
        # Secondary precipitation variables from CHIRPS
        "chirps_wet_day_count":      f"{R}/chirps/chirps_wet_day_count_{L}.tif",
        "chirps_monthly_std":        f"{R}/chirps/chirps_monthly_std_{L}.tif",

        #GLDAS weather variables
        'gldas_air_temp':   f"{R}/gldas_air_temp/gldas_air_temp_monthly_mean_{L}.tif",
        'gldas_lwdown': f"{R}/gldas_lwdown/gldas_lwdown_monthly_sum_{L}.tif",
        'gldas_specific_humidity': f"{R}/gldas_specific_humidity/gldas_specific_humidity_monthly_mean_{L}.tif",
        'gldas_surface_pressure': f"{R}/gldas_surface_pressure/gldas_surface_pressure_monthly_mean_{L}.tif",
        'gldas_swdown': f"{R}/gldas_swdown/gldas_swdown_monthly_sum_{L}.tif",

        # MODIS EVI
        "evi_monthly_mean":         f"{R}/modis_evi/evi_monthly_mean_{L}.tif",
        "evi_missing_fraction":     f"{R}/modis_evi/evi_missing_fraction_{L}.tif",

        # MODIS LST
        "lst_d_monthly_mean":       f"{R}/modis_lst/lst_d_monthly_mean_{L}.tif",
        "lst_d_missing_fraction":   f"{R}/modis_lst/lst_d_missing_fraction_{L}.tif",
        "lst_n_monthly_mean":       f"{R}/modis_lst/lst_n_monthly_mean_{L}.tif",
        "lst_n_missing_fraction":   f"{R}/modis_lst/lst_n_missing_fraction_{L}.tif",
        "lst_diff_monthly_mean":    f"{R}/modis_lst/lst_diff_monthly_mean_{L}.tif",
        "lst_diff_missing_fraction":f"{R}/modis_lst/lst_diff_missing_fraction_{L}.tif"

        #Kriged Insitu data
#        "insitu_kriged":   f"{R}/Insitu_kriged/insitu_precip_{L}.tif"
    }

In [5]:
# run the main month loop

month_range = pd.period_range(
    start = f"{START_YEAR}-{START_MONTH:02d}",
    end = f"{END_YEAR}-{END_MONTH:02d}",
    freq = "M"
)

print(f"\nProcessing {len(month_range)} months.. \n")

for period in tqdm(month_range, desc = "Monthly feature extraction"):
    year = period.year
    month = period.month
    label = mlabel(year, month)

    pred_out = os.path.join(MONTHLY_PRED_DIR, f"Merged_{label}.csv")

    if os.path.exists(pred_out):
        continue

    paths = build_monthly_paths(year, month)
    aligned = []
    missing_vars = []

    for name, path in paths.items():
        if not os.path.exists(path):
            missing_vars.append(name)
            continue
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop = True)
        da.name = name
        aligned.append(da.load())
        da.close()

    if missing_vars:
        tqdm.write(f" [{label}] missing rasters: {missing_vars}")
    
    if not aligned:
        tqdm.write(f" [{label}] no rasters found - skipping")
        continue

    dynamic_xr = xr.merge(aligned, compat='override')
    dynamic_df = dynamic_xr.to_dataframe().reset_index()

    # variable without rasters gets Nan Column
    for name in missing_vars:
        dynamic_df[name] = np.nan

    del aligned, dynamic_xr
    gc.collect()

    #merge with static df
    monthly_grid = dynamic_df.merge(static_df, on = ['x', 'y'], how = "inner")

    #Add temporal features - cyclical month columns
    sin_m, cos_m = month_cyclical(month)
    monthly_grid['month_sin'] = sin_m
    monthly_grid['month_cos'] = cos_m
    monthly_grid['year'] = year
    monthly_grid['month'] = month

    monthly_grid.to_csv(pred_out, index=False)

    del monthly_grid
    gc.collect()



Processing 60 months.. 



Monthly feature extraction: 100%|██████████| 60/60 [10:32<00:00, 10.55s/it]


In [7]:
# aggregate monthly csvs to yearly parquet -- to final combined df 

def aggregate_to_parquet(monthly_dir: str, parquet_dir: str) -> str:
    """
    Read monthly CSVs year by year, write to parquet, then concatenate
    all years into a single final df. Returns path to final parquet.
    """
    parquet_paths = []

    for year in range(START_YEAR, END_YEAR + 1):
        yearly_path = os.path.join(parquet_dir, f"full_{year}.parquet")
        monthly_periods = pd.period_range(
            start = f"{year}-01", end = f"{year}-12", freq = "M"
        )
        expected = [
            os.path.join(monthly_dir, f"merged_{mlabel(p.year, p.month)}.csv")
            for p in monthly_periods
        ]
        found = [f for f in expected if os.path.exists(f)]

        if not found:
            print(f"{year} - no monthly files found, skipping")
            continue

        writer = None
        for fpath in found:
            chunk = pd.read_csv(fpath)
            table = pa.Table.from_pandas(chunk)
            if writer is None:
                writer = pq.ParquetWriter(yearly_path, table.schema)
            writer.write_table(table)
        
        if writer:
            writer.close()
            parquet_paths.append(yearly_path)
            print(f"{year} - {len(found)} months written to parquet")
    
    if not parquet_paths:
        print(f" No data - final df not written")
        return None 
    
    final_df = pd.concat([pd.read_parquet(p) for p in parquet_paths], ignore_index = True)
    final_path = os.path.join(OUT_BASE, f"final_df.parquet")
    final_df.to_parquet(final_path, index=False)

    final_df.head(10_000).to_csv(
        os.path.join(OUT_BASE, f"final_df_preview.csv"), index=False
    )
 
    print(f"\n  Final df: {len(final_df):,} rows × {len(final_df.columns)} cols")
    print(f"  Saved: {final_path}")
    return final_path

print("Aggregating to yearly parquet files")

pred_final_path = aggregate_to_parquet(MONTHLY_PRED_DIR, PARQUET_PRED_DIR)

Aggregating to yearly parquet files
2001 - 12 months written to parquet
2002 - 12 months written to parquet
2003 - 12 months written to parquet
2004 - 12 months written to parquet
2005 - 12 months written to parquet

  Final df: 29,077,140 rows × 41 cols
  Saved: ..\Data\Processed\PredictedRasters\final_df.parquet


In [8]:
import shutil

print("\n" + "=" * 60)
print("Cleaning up intermediate files and directories...")
print("=" * 60)

dirs_to_remove = [
    MONTHLY_PRED_DIR,
    PARQUET_PRED_DIR
]

for d in dirs_to_remove:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed directory: {d}")


Cleaning up intermediate files and directories...
Removed directory: ..\Data\Processed\PredictedRasters\monthly_dynamic_pred_df
Removed directory: ..\Data\Processed\PredictedRasters\yearly_parquet_pred
